# Akanshya Chakraborty (230953042): Deep Learning Mini Project (ICT-4442), Team 06
**Comparative Study of Deep Learning Architectures for Automated Waste Classification**

This notebook covers: **Model 4: ViT-B/16** and the single comparison table for all four models.

**How to run** (about 30 to 40 min, plus 1 min for the comparison cell at the end on a free T4):
1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > **Run all**. The dataset downloads with `kagglehub`; if that fails you are asked to upload `kaggle.json` (Kaggle > Settings > API > Create New Token).
3. At the end a zip `Akanshya_Chakraborty_results.zip` downloads. Unzip it inside your local clone of the repository so that `Akanshya_Chakraborty/results/` appears next to this notebook, then commit and push (see the repository README).

All four members use the same split (seed 42), preprocessing, augmentation, loss and evaluation code. The line `Split fingerprint` printed after the split must be identical in all four notebooks.

In [ ]:
MEMBER = "Akanshya_Chakraborty"
DO_EDA = False   # only the dataset owner produces the EDA figures

## Setup

In [ ]:
# ============ CELL 1: setup ============
import os, sys, time, json, math, copy, random, subprocess, base64, io, datetime, warnings
warnings.filterwarnings("ignore")
SMOKE = globals().get("SMOKE", False)          # True = tiny self-test mode (leave False on Colab)
if not SMOKE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx", "kagglehub"], check=False)

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
MEMBER = globals().get("MEMBER", "member")
OUT = f"/content/{MEMBER}" if os.path.isdir("/content") else os.path.abspath(MEMBER)
RES = os.path.join(OUT, "results"); FIG = os.path.join(RES, "figures")
os.makedirs(FIG, exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev == "cpu" and not SMOKE:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
GPU_NAME = torch.cuda.get_device_name(0) if dev == "cuda" else "CPU (self-test)"
use_amp = dev == "cuda"
torch.backends.cudnn.benchmark = True

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if dev == "cuda": torch.cuda.manual_seed_all(s)
set_seed()

# ---- training configuration (one fixed recipe per model) ----
if SMOKE:
    CFG = dict(mlp_ep=2, cnn_ep=2, res_head_ep=1, res_ft_ep=1, vit_ep=1,
               bs_mlp=16, bs_cnn=16, bs_res=16, bs_vit=16, lat_n=5, lat_warm=2)
else:
    CFG = dict(mlp_ep=40, cnn_ep=30, res_head_ep=3, res_ft_ep=7, vit_ep=5,
               bs_mlp=128, bs_cnn=64, bs_res=64, bs_vit=32, lat_n=100, lat_warm=20)
print("Device:", GPU_NAME, "| torch", torch.__version__, "| SMOKE =", SMOKE)


## Dataset

In [ ]:
# ============ CELL 2: get the dataset + cache images ============
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def find_root(base):
    for dp, dn, fn in os.walk(base):
        good = [d for d in dn if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(dp, d)))]
        if len(good) >= 12:
            return dp
    return None

def download_dataset():
    try:
        import kagglehub
        p = kagglehub.dataset_download("mostafaabla/garbage-classification")
        r = find_root(p)
        if r: return r
    except Exception as e:
        print("kagglehub failed:", repr(e)[:200])
    print("\nUpload your kaggle.json (Kaggle > Settings > API > Create New Token) when prompted.")
    from google.colab import files
    up = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f: f.write(up[list(up)[0]])
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("/content/data", exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "mostafaabla/garbage-classification",
                    "-p", "/content/data", "--unzip"], check=True)
    return find_root("/content/data")

DATA_ROOT = globals().get("DATA_ROOT") or download_dataset()
classes = sorted(d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d))
                 and any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(DATA_ROOT, d))))
assert len(classes) == 12, f"expected 12 classes, found {classes}"
K = len(classes)
paths, labels = [], []
for ci, c in enumerate(classes):
    for f in sorted(os.listdir(os.path.join(DATA_ROOT, c))):
        if f.lower().endswith(IMG_EXT):
            paths.append(os.path.join(DATA_ROOT, c, f)); labels.append(ci)
labels = np.array(labels)
print(K, "classes,", len(paths), "image files found")

def load_one(p):
    try:
        im = Image.open(p)
        im.draft("RGB", (448, 448))
        im = im.convert("RGB").resize((224, 224), Image.BILINEAR)
        return np.asarray(im).transpose(2, 0, 1)
    except Exception:
        return None

t0 = time.time()
arr = np.zeros((len(paths), 3, 224, 224), dtype=np.uint8)
valid = np.ones(len(paths), dtype=bool)
with ThreadPoolExecutor(8) as ex:
    for i, a in enumerate(ex.map(load_one, paths)):
        if a is None: valid[i] = False
        else: arr[i] = a
N_BAD = int((~valid).sum())
if N_BAD:
    arr = arr[valid]; labels = labels[valid]; paths = [p for p, v in zip(paths, valid) if v]
X = torch.from_numpy(arr); del arr
Y = torch.from_numpy(labels).long()
N = len(Y)
print(f"Loaded {N} images ({N_BAD} unreadable files dropped) in {time.time()-t0:.0f}s; tensor {tuple(X.shape)} uint8")


## Split, class weights, near-duplicate check

In [ ]:
# ============ CELL 3: stratified split, class weights, EDA, near-duplicate check ============
import pandas as pd
from sklearn.model_selection import train_test_split
idx = np.arange(N)
tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=42)
va, te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=42)
tr_t, va_t, te_t = [torch.from_numpy(a).long() for a in (tr, va, te)]
split_df = pd.DataFrame({"path": [os.path.relpath(p, DATA_ROOT) for p in paths], "label": labels,
                         "class": [classes[l] for l in labels], "split": "train"})
split_df.loc[va, "split"] = "val"; split_df.loc[te, "split"] = "test"
split_df.to_csv(os.path.join(RES, "split_seed42.csv"), index=False)
json.dump({c: i for i, c in enumerate(classes)}, open(os.path.join(RES, "class_to_index.json"), "w"), indent=1)
print("train/val/test =", len(tr), len(va), len(te))

cnt_all = np.bincount(labels, minlength=K); cnt_tr = np.bincount(labels[tr], minlength=K)
cnt_va = np.bincount(labels[va], minlength=K); cnt_te = np.bincount(labels[te], minlength=K)
CLS_W = torch.tensor(len(tr) / (K * np.maximum(cnt_tr, 1)), dtype=torch.float32)

# four disposal streams (our grouping; edit here if your team defined it differently)
STREAM_OF = {"battery": "Hazardous", "biological": "Organic", "brown-glass": "Recyclable", "cardboard": "Recyclable",
             "clothes": "Residual/textile", "green-glass": "Recyclable", "metal": "Recyclable", "paper": "Recyclable",
             "plastic": "Recyclable", "shoes": "Residual/textile", "trash": "Residual/textile", "white-glass": "Recyclable"}
STREAMS = ["Recyclable", "Organic", "Hazardous", "Residual/textile"]
if SMOKE: STREAM_OF = {c: STREAMS[i % 4] for i, c in enumerate(classes)}
assert set(classes) == set(STREAM_OF), set(classes) ^ set(STREAM_OF)
C2S = np.array([STREAMS.index(STREAM_OF[c]) for c in classes])

DO_EDA = globals().get('DO_EDA', False)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110})
if DO_EDA:
    # --- EDA figures ---
    order = np.argsort(-cnt_all)
    fig, ax = plt.subplots(figsize=(6.5, 3))
    ax.bar([classes[i] for i in order], cnt_all[order], color="#4C78A8")
    ax.set_ylabel("Images"); plt.xticks(rotation=40, ha="right")
    for x_, i in enumerate(order): ax.text(x_, cnt_all[i], str(cnt_all[i]), ha="center", va="bottom", fontsize=7)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_class_distribution.png", dpi=200); plt.show()

    fig, axs = plt.subplots(2, 6, figsize=(6.5, 2.6))
    for c, ax in enumerate(axs.ravel()):
        j = np.where(labels == c)[0][0]
        ax.imshow(X[j].permute(1, 2, 0).numpy()); ax.set_title(classes[c], fontsize=7); ax.axis("off")
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_samples.png", dpi=200); plt.show()

# --- near-duplicate check (64-bit difference hash, Hamming distance <= 4) ---
def dhash_all(Xu):
    out = []
    for i in range(0, len(Xu), 512):
        g = Xu[i:i+512].float().mean(1, keepdim=True)
        g = F.interpolate(g, size=(8, 9), mode="area")
        out.append((g[..., 1:] > g[..., :-1]).reshape(len(g), 64))
    return torch.cat(out).float()
H = dhash_all(X)
def min_ham(a, b):
    res = []
    for i in range(0, len(a), 256):
        aa = H[a[i:i+256]]; bb = H[b]
        d = aa @ (1 - bb).T + (1 - aa) @ bb.T
        res.append(d.min(1).values)
    return torch.cat(res).numpy()
DUP_THR = 4
dist_te = min_ham(te_t, tr_t); dist_va = min_ham(va_t, tr_t)
te_dup = dist_te <= DUP_THR; va_dup = dist_va <= DUP_THR
print(f"Near-duplicates of a training image: test {te_dup.sum()}/{len(te)}, val {va_dup.sum()}/{len(va)}")

import hashlib
SPLIT_FINGERPRINT = hashlib.md5(np.concatenate([tr, va, te]).tobytes()).hexdigest()[:12]
print("Split fingerprint (must be identical for all four members):", SPLIT_FINGERPRINT, "| N =", N)
if DO_EDA:
    json.dump(dict(n_images=int(N), n_unreadable_dropped=int(N_BAD), class_counts=dict(zip(classes, map(int, cnt_all))),
                   n_train=len(tr), n_val=len(va), n_test=len(te), dedup_hash="64-bit dHash", dedup_hamming_threshold=DUP_THR,
                   near_dup_in_test=int(te_dup.sum()), near_dup_in_val=int(va_dup.sum()), split_fingerprint=SPLIT_FINGERPRINT),
              open(os.path.join(RES, "dataset_summary.json"), "w"), indent=1)


## Training loop and shared evaluation code

In [ ]:
# ============ CELL 4: GPU augmentation, training loop, evaluation ============
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
_T = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]], device=dev)
_Ti = torch.linalg.inv(_T)

def augment(x):
    """flip, rotation +-15 deg, brightness/contrast/saturation 0.8-1.2, small hue shift (+-0.03 turn). x in [0,1]."""
    B, d = x.shape[0], x.device
    x = torch.where((torch.rand(B, device=d) < 0.5)[:, None, None, None], x.flip(3), x)
    a = (torch.rand(B, device=d) * 2 - 1) * math.radians(15)
    th = torch.zeros(B, 2, 3, device=d)
    th[:, 0, 0] = torch.cos(a); th[:, 0, 1] = -torch.sin(a); th[:, 1, 0] = torch.sin(a); th[:, 1, 1] = torch.cos(a)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="reflection", align_corners=False)
    r = lambda lo, hi: torch.rand(B, 1, 1, 1, device=d) * (hi - lo) + lo
    x = x * r(.8, 1.2)
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * r(.8, 1.2) + m
    g = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]; x = (x - g) * r(.8, 1.2) + g
    h = (torch.rand(B, device=d) * 2 - 1) * 0.03 * 2 * math.pi
    R = torch.zeros(B, 3, 3, device=d); R[:, 0, 0] = 1
    R[:, 1, 1] = torch.cos(h); R[:, 1, 2] = -torch.sin(h); R[:, 2, 1] = torch.sin(h); R[:, 2, 2] = torch.cos(h)
    M = _Ti @ R @ _T
    x = torch.einsum("bij,bjhw->bihw", M, x)
    return x.clamp(0, 1)

def prep(xb, size, norm, aug, cl=False):
    x = xb.to(dev, non_blocking=True).float() / 255
    if aug: x = augment(x)
    if size != 224: x = F.interpolate(x, size=(size, size), mode="bilinear", antialias=True, align_corners=False)
    if norm: x = (x - MEAN) / STD
    return x.contiguous(memory_format=torch.channels_last) if cl else x

@torch.no_grad()
def predict(model, idx, size, norm, cl=False, bs=128):
    model.eval(); outs = []
    for i in range(0, len(idx), bs):
        ib = idx[i:i+bs]
        with torch.autocast(device_type=dev, dtype=torch.float16, enabled=use_amp):
            outs.append(model(prep(X[ib], size, norm, False, cl)).float().cpu())
    return torch.cat(outs)

def metrics(y, p):
    P, R, F1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return dict(acc=float(accuracy_score(y, p)), p=float(P), r=float(R), f1=float(F1),
                per_class_f1=[float(v) for v in f1_score(y, p, average=None, labels=list(range(K)), zero_division=0)],
                cm=confusion_matrix(y, p, labels=list(range(K))).tolist())

def stream_metrics(y, p):
    ys, ps = C2S[y], C2S[p]
    return dict(acc=float(accuracy_score(ys, ps)), f1=float(f1_score(ys, ps, average="macro", zero_division=0)))

def fit(name, model, size, norm, bs, phases, aug=True, cl=False):
    model.to(dev)
    if cl: model.to(memory_format=torch.channels_last)
    crit_w = CLS_W.to(dev); yv = Y[va_t].numpy()
    best = dict(f1=-1, state=None, epoch=-1); hist = []; t0 = time.time(); ep_g = 0
    for ph in phases:
        for p in model.parameters(): p.requires_grad = False
        groups = []
        for g in ph["groups"]:
            ps = list(g["params"])
            for p in ps: p.requires_grad = True
            groups.append({"params": ps, "lr": g["lr"], "weight_decay": ph["wd"]})
        opt = torch.optim.AdamW(groups)
        spe = math.ceil(len(tr_t) / bs); total = spe * ph["epochs"]; warm = max(1, int(0.05 * total))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                                  else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        for ep in range(ph["epochs"]):
            model.eval()
            for m in ph["train_modules"]: m.train()
            perm = tr_t[torch.randperm(len(tr_t))]; run = 0.0; nb = 0
            for i in range(0, len(perm), bs):
                ib = perm[i:i+bs]
                if len(ib) < 2: continue
                x = prep(X[ib], size, norm, aug, cl); y = Y[ib].to(dev)
                with torch.autocast(device_type=dev, dtype=torch.float16, enabled=use_amp):
                    loss = F.cross_entropy(model(x).float(), y, weight=crit_w)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                if ph.get("clip"):
                    scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), ph["clip"])
                scaler.step(opt); scaler.update(); sched.step()
                run += loss.item(); nb += 1
            lg = predict(model, va_t, size, norm, cl); pv = lg.argmax(1).numpy()
            vl = float(F.cross_entropy(lg, Y[va_t], weight=CLS_W)); vf = float(f1_score(yv, pv, average="macro", zero_division=0))
            ep_g += 1
            hist.append(dict(epoch=ep_g, train_loss=run / max(nb, 1), val_loss=vl, val_acc=float(accuracy_score(yv, pv)), val_f1=vf))
            print(f"  [{name}] ep {ep_g:2d} | train loss {run/max(nb,1):.3f} | val loss {vl:.3f} | val acc {hist[-1]['val_acc']:.3f} | val macro-F1 {vf:.3f} | {time.time()-t0:.0f}s", flush=True)
            if vf > best["f1"]:
                best = dict(f1=vf, epoch=ep_g, state={k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()})
    train_time = time.time() - t0
    model.load_state_dict(best["state"])
    return dict(hist=hist, best_val_f1=best["f1"], best_epoch=best["epoch"], train_time_s=train_time)

@torch.no_grad()
def latency_ms(model, size, norm, cl=False):
    model.eval(); n, w = CFG["lat_n"], CFG["lat_warm"]
    xs = [prep(X[te_t[i:i+1]], size, norm, False, cl) for i in range(n + w)]
    for i in range(n + w):
        if i == w:
            if dev == "cuda": torch.cuda.synchronize()
            t = time.perf_counter()
        with torch.autocast(device_type=dev, dtype=torch.float16, enabled=use_amp): model(xs[i])
    if dev == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000

RESULTS, PRED = {}, {}
def run_model(key, disp, family, builder, size, norm, bs, phases_fn, aug=True, cl=False):
    print(f"\n=== {disp} ===", flush=True)
    set_seed(SEED)
    model = builder(); params = sum(p.numel() for p in model.parameters())
    info = fit(disp, model, size, norm, bs, phases_fn(model), aug, cl)
    lg = predict(model, te_t, size, norm, cl); pr = F.softmax(lg, 1)
    yt = Y[te_t].numpy(); pp = lg.argmax(1).numpy()
    RESULTS[key] = dict(key=key, name=disp, family=family, params=params, aug=aug, input=size, member=MEMBER,
                        latency_ms=latency_ms(model, size, norm, cl), test=metrics(yt, pp), stream=stream_metrics(yt, pp),
                        classes=classes, streams=STREAMS, split_fingerprint=SPLIT_FINGERPRINT, seed=SEED, gpu=GPU_NAME,
                        torch=torch.__version__, config={k: v for k, v in CFG.items()}, **info)
    PRED[key] = dict(pred=pp, conf=pr.max(1).values.numpy())
    r = RESULTS[key]
    print(f"  TEST: acc {r['test']['acc']:.4f} | macro-F1 {r['test']['f1']:.4f} | stream acc {r['stream']['acc']:.4f} | "
          f"{params/1e6:.2f}M params | {r['train_time_s']/60:.1f} min | {r['latency_ms']:.2f} ms/img", flush=True)
    json.dump(r, open(os.path.join(RES, f"{key}_results.json"), "w"), indent=1)
    np.savez(os.path.join(RES, f"{key}_test_predictions.npz"), y_true=yt, pred=pp, conf=PRED[key]["conf"])
    del model
    if dev == "cuda": torch.cuda.empty_cache()


## Model definition

In [ ]:
# ============ Model code (this member's model only) ============
import torchvision
PRETRAINED = not SMOKE

from torchvision.models import vit_b_16, ViT_B_16_Weights
def build_vit():
    m = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1 if PRETRAINED else None)
    m.heads = nn.Linear(768, K); return m
def phases_vit(m):
    return [dict(epochs=CFG["vit_ep"], wd=0.05, clip=1.0,
                 groups=[dict(params=[p for n, p in m.named_parameters() if not n.startswith("heads")], lr=3e-5),
                         dict(params=m.heads.parameters(), lr=3e-4)], train_modules=[m])]


## Train and test

In [ ]:
# ============ Train + test ViT-B/16 (slowest, about 15-25 min on a T4) ============
run_model("vit", "ViT-B/16", "Transformer", build_vit, 224, True, CFG["bs_vit"], phases_vit)


## Figures, summary and zip for GitHub

In [ ]:
# ============ Figures + summary for this member's model(s), then zip for GitHub ============
import shutil
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
yt = Y[te_t].numpy()
COL = {"mlp": "#E69F00", "cnn": "#009E73", "resnet": "#0072B2", "vit": "#CC79A7", "cnn_noaug": "#999999"}
for key, r in RESULTS.items():
    h = r["hist"]
    fig, ax = plt.subplots(1, 2, figsize=(6.5, 2.6))
    ax[0].plot([e["epoch"] for e in h], [e["train_loss"] for e in h], label="train"); ax[0].plot([e["epoch"] for e in h], [e["val_loss"] for e in h], label="val")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss (class-weighted CE)"); ax[0].legend(frameon=False, fontsize=7)
    ax[1].plot([e["epoch"] for e in h], [e["val_f1"] for e in h], color=COL[key]); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation macro-F1")
    plt.suptitle(r["name"], fontsize=9); plt.tight_layout(); plt.savefig(f"{FIG}/{key}_curves.png", dpi=200); plt.show()

    cm = np.array(r["test"]["cm"]); cn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.6, 5.0)); im = ax.imshow(cn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(classes, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(classes, fontsize=7)
    for i in range(K):
        for j in range(K):
            if cn[i, j] >= 0.01: ax.text(j, i, f"{cn[i,j]:.2f}", ha="center", va="center", fontsize=5.5, color="white" if cn[i, j] > .5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{r['name']}: row-normalised confusion matrix", fontsize=9)
    plt.colorbar(im, fraction=0.046, pad=0.03); plt.tight_layout(); plt.savefig(f"{FIG}/{key}_confusion.png", dpi=200); plt.show()

    fig, ax = plt.subplots(figsize=(6.0, 2.8)); v = r["test"]["per_class_f1"]
    ax.bar(classes, v, color=COL[key]); ax.set_ylim(0, 1.05); ax.set_ylabel("Test F1"); plt.xticks(rotation=40, ha="right")
    for i, vv in enumerate(v): ax.text(i, vv, f"{vv:.2f}", ha="center", va="bottom", fontsize=6)
    ax.set_title(f"{r['name']}: per-class F1", fontsize=9); plt.tight_layout(); plt.savefig(f"{FIG}/{key}_per_class_f1.png", dpi=200); plt.show()

    pb, cf = PRED[key]["pred"], PRED[key]["conf"]; wrong = np.where(pb != yt)[0]; worst = wrong[np.argsort(-cf[wrong])][:8]
    if len(worst):
        fig, axs = plt.subplots(2, 4, figsize=(6.5, 3.6))
        for ax in axs.ravel(): ax.axis("off")
        for ax, w_ in zip(axs.ravel(), worst):
            ax.imshow(X[te_t[w_]].permute(1, 2, 0).numpy()); ax.set_title(f"True: {classes[yt[w_]]}\nPred: {classes[pb[w_]]} ({cf[w_]:.2f})", fontsize=6.5)
        plt.tight_layout(); plt.savefig(f"{FIG}/{key}_most_confident_errors.png", dpi=200); plt.show()

# summary table
lines = [f"# {MEMBER.replace('_', ' ')}: results", "", f"Split fingerprint: `{SPLIT_FINGERPRINT}` | GPU: {GPU_NAME} | seed {SEED}", "",
         "| Model | Test acc | Macro-P | Macro-R | Macro-F1 | 4-stream acc | Params (M) | Train (min) | Latency (ms) | Best val F1 (epoch) |", "|---|---|---|---|---|---|---|---|---|---|"]
for key, r in RESULTS.items():
    t = r["test"]
    lines.append(f"| {r['name']} | {t['acc']:.4f} | {t['p']:.4f} | {t['r']:.4f} | {t['f1']:.4f} | {r['stream']['acc']:.4f} | {r['params']/1e6:.2f} | {r['train_time_s']/60:.1f} | {r['latency_ms']:.2f} | {r['best_val_f1']:.4f} ({r['best_epoch']}) |")
if "cnn" in RESULTS and "cnn_noaug" in RESULTS:
    d = RESULTS["cnn"]["test"]["f1"] - RESULTS["cnn_noaug"]["test"]["f1"]
    lines += ["", f"Augmentation ablation: macro-F1 with augmentation minus without = **{d:+.4f}**."]
    fig, ax = plt.subplots(figsize=(4.2, 2.6)); w = .35
    for j, (nm, key) in enumerate([("With augmentation", "cnn"), ("Without augmentation", "cnn_noaug")]):
        vv = [RESULTS[key]["test"]["acc"], RESULTS[key]["test"]["f1"]]
        ax.bar(np.arange(2) + (j - .5) * w, vv, w, label=nm, color=["#009E73", "#999999"][j])
        for x_, v_ in zip(np.arange(2) + (j - .5) * w, vv): ax.text(x_, v_, f"{v_:.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(range(2)); ax.set_xticklabels(["Accuracy", "Macro-F1"]); ax.set_ylim(0, 1.05); ax.legend(fontsize=7, frameon=False, loc="lower right")
    plt.tight_layout(); plt.savefig(f"{FIG}/augmentation_ablation.png", dpi=200); plt.show()
lines += ["", "Worst classes by test F1: " + "; ".join(
    f"{r['name']}: " + ", ".join(f"{classes[i]} ({r['test']['per_class_f1'][i]:.2f})" for i in np.argsort(r['test']['per_class_f1'])[:3]) for r in RESULTS.values())]
open(os.path.join(RES, f"README_{MEMBER}.md"), "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

zip_base = f"/content/{MEMBER}_results" if os.path.isdir("/content") else os.path.join(OUT, "..", f"{MEMBER}_results")
shutil.make_archive(zip_base, "zip", os.path.dirname(OUT), os.path.basename(OUT))
print("\nZip created:", zip_base + ".zip", "(unzip it inside your local repo clone so that the folder", MEMBER + "/results/ appears)")
try:
    from google.colab import files; files.download(zip_base + ".zip")
except Exception as e:
    print("Download from the Files panel (folder icon) instead:", zip_base + ".zip")


## Comparison table for all four models
Run this cell **after the other three members have pushed to `main`**. It clones the repository, reads every member's `results/*_results.json`, and writes `comparison/` (table + figures) into your results folder. Push those too.

In [ ]:
# ============ Comparison table across the four members' results (run AFTER teammates have pushed to main) ============
import glob, pandas as pd
REPO_URL = "https://github.com/Akanshya1595/DL_Project"
REPO_DIR = globals().get("REPO_DIR") or "/content/DL_Project_repo"
if not globals().get("REPO_DIR"):
    subprocess.run(["rm", "-rf", REPO_DIR]); subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
ALL = {}
for fn in glob.glob(f"{REPO_DIR}/*/results/*_results.json"):
    r = json.load(open(fn)); ALL[r["key"]] = r
for k, r in RESULTS.items(): ALL[k] = r                    # use this notebook's own run for ViT
ORDER = [k for k in ["mlp", "cnn", "resnet", "vit"] if k in ALL]
print("Found results for:", ORDER, "| missing:", [k for k in ["mlp", "cnn", "resnet", "vit"] if k not in ALL])
fps = {k: ALL[k].get("split_fingerprint") for k in ALL}
print("Split fingerprints:", fps, "<- all must be identical")
CMP = os.path.join(RES, "comparison"); os.makedirs(CMP, exist_ok=True)
rows = []
for k in ORDER + (["cnn_noaug"] if "cnn_noaug" in ALL else []):
    r = ALL[k]; t = r["test"]
    rows.append({"Model": r["name"], "Owner": r.get("member", "").replace("_", " "), "Accuracy": round(t["acc"], 4), "Macro-P": round(t["p"], 4), "Macro-R": round(t["r"], 4),
                 "Macro-F1": round(t["f1"], 4), "4-stream acc": round(r["stream"]["acc"], 4), "Params (M)": round(r["params"] / 1e6, 2),
                 "Train (min)": round(r["train_time_s"] / 60, 1), "Latency (ms)": round(r["latency_ms"], 2)})
df = pd.DataFrame(rows); df.to_csv(os.path.join(CMP, "comparison_table.csv"), index=False)
try: open(os.path.join(CMP, "comparison_table.md"), "w").write(df.to_markdown(index=False))
except Exception: open(os.path.join(CMP, "comparison_table.md"), "w").write(df.to_string(index=False))
print(df.to_string(index=False))

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
cls_ = ALL[ORDER[0]]["classes"]; M = np.array([ALL[k]["test"]["per_class_f1"] for k in ORDER]).T
fig, ax = plt.subplots(figsize=(4.6, 4.0)); im = ax.imshow(M, cmap="YlGnBu", vmin=max(0, M.min() - .05), vmax=1)
ax.set_xticks(range(len(ORDER))); ax.set_xticklabels([ALL[k]["name"] for k in ORDER], fontsize=7, rotation=30, ha="right"); ax.set_yticks(range(len(cls_))); ax.set_yticklabels(cls_, fontsize=7)
for i in range(len(cls_)):
    for j in range(len(ORDER)): ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if M[i, j] > .85 else "black")
plt.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.savefig(f"{CMP}/per_class_f1_all_models.png", dpi=200); plt.show()
C_ = {"mlp": "#E69F00", "cnn": "#009E73", "resnet": "#0072B2", "vit": "#CC79A7"}
fig, ax = plt.subplots(figsize=(4.8, 3.2))
for k in ORDER:
    r = ALL[k]; ax.scatter(r["latency_ms"], r["test"]["f1"], s=30 + r["params"] / 1e6 * 4, color=C_[k])
    ax.annotate(f'{r["name"]} ({r["params"]/1e6:.1f}M)', (r["latency_ms"], r["test"]["f1"]), textcoords="offset points", xytext=(6, 5), fontsize=7)
ax.set_xlabel("Latency per image, batch 1 (ms)"); ax.set_ylabel("Test macro-F1"); ax.margins(x=.25, y=.15)
plt.tight_layout(); plt.savefig(f"{CMP}/tradeoff_f1_vs_latency.png", dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(5.6, 2.8))
for k in ORDER: ax.plot([e["epoch"] for e in ALL[k]["hist"]], [e["val_f1"] for e in ALL[k]["hist"]], color=C_[k], label=ALL[k]["name"])
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation macro-F1"); ax.legend(frameon=False, fontsize=7); plt.tight_layout(); plt.savefig(f"{CMP}/val_f1_curves_all_models.png", dpi=200); plt.show()
import shutil
shutil.make_archive(zip_base, "zip", os.path.dirname(OUT), os.path.basename(OUT))     # re-zip with the comparison folder included
try:
    from google.colab import files; files.download(zip_base + ".zip")
except Exception: pass
